In [ ]:
# import funcs
%run ./utilsMassProfile.ipynb

from scipy.stats import binned_statistic

In [4]:
def step_to_skycoord(step_df):
    gc = SkyCoord(
        x=step_df["X"].values * u.pc,
        y=step_df["Y"].values * u.pc,
        z=step_df["Z"].values * u.pc,
        v_x=step_df["Vx"].values * (u.km/u.s),
        v_y=step_df["Vy"].values * (u.km/u.s),
        v_z=step_df["Vz"].values * (u.km/u.s),
        frame=Galactocentric()
    )
    return gc

# this didnt work very well, swapped to using std away from mean vel
def filter_stream_old(step_df, density_filter=20, bins=50):
    density_filter = density_filter * u.solMass / u.deg**2
    
    icrs = step_to_skycoord(step_df).icrs
    
    mass_grid, xedges, yedges = np.histogram2d(icrs.ra, icrs.dec,
                                               bins=bins,weights=step_df['M'])
    
    dx = (xedges[1] - xedges[0])
    dy = (yedges[1] - yedges[0])
    density = mass_grid * u.solMass / (dx * dy)
    mask = density > density_filter

    xidx = np.digitize(icrs.ra, xedges[1:-1])
    yidx = np.digitize(icrs.dec, yedges[1:-1])
    
    xidx = np.clip(xidx, 0, bins - 1)
    yidx = np.clip(yidx, 0, bins - 1)
    
    row_mask = mask[xidx, yidx]
    
    stream_df = step_df[row_mask]
    return stream_df

def filter_stream(step_df, sigma=0.5):
    Vel    = np.column_stack((step_df['Vx'],step_df['Vy'],step_df['Vz']))
    centre_v = np.median(Vel, axis=0)
    Vel  -= centre_v

    speed = np.linalg.norm(Vel, axis=1)
    std_speed = np.std(speed)

    mask = speed < std_speed * sigma
    return step_df[mask]
    

    


def step_to_stream_coords(stream_df):
    icrs = step_to_skycoord(stream_df).icrs

    x,y,z = shrinking_sphere(*stream_df[['X', 'Y', 'Z', 'M']].T.to_numpy())
    centre_deg = SkyCoord(x=x * u.pc,
                          y=y * u.pc,
                          z=z * u.pc,
                         frame=Galactocentric()
                    ).icrs 

    # finding rotation between average ra dec velocity (orbital) and x axis 
    #- https://stackoverflow.com/questions/6247153/angle-from-2d-unit-vector
    pmra_mean = np.median(icrs.pm_ra_cosdec.to_value())
    pmdec_mean = np.median(icrs.pm_dec.to_value())
    
    theta = np.arctan2(pmdec_mean, pmra_mean) * u.rad
    theta = theta.to(u.deg)
    tp = icrs.transform_to(centre_deg.skyoffset_frame(rotation=-theta))
    
    stream_df["phi1"] = tp.lon.deg
    stream_df["phi2"] = tp.lat.deg
    return tp


    

In [ ]:
def steam_density(stream_df, phi1_lim=5, bins=50):
    
    phi1, phi2 = stream_df['phi1'], stream_df['phi2']
    m = stream_df['M']

    w = m / (2*phi1_lim/bins)
    hist, edges = np.histogram(phi1, bins=bins, weights=w, range=[-phi1_lim,phi1_lim])
    return hist, edges


def stream_length(stream_df):
    return np.max(stream_df['phi1']) - np.min(stream_df['phi1'])

def stream_width(stream_df, bins=40):
    stds, bin_edges, binnumber =  binned_statistic(stream_df['phi1'], stream_df['phi2'], statistic='std', bins=bins)
    # filter out NaNs
    stds = np.asarray(stds, dtype=float) 
    stds_clean = np.nan_to_num(stds, nan=0.0, posinf=0.0, neginf=0.0)
    return stds_clean, bin_edges, binnumber


    
    
    
    